# Healthcare Billing & Insurance Dataset — Data Cleaning & Preparation

## Introduction & Objective
This notebook documents the data quality audit, cleaning, and preparation of `healthcare_dataset.csv`, resulting in clean records across 15 columns.

### Cleaning & Preparation Steps Covered:
* **Schema & Text Standardization:** Standardized column headers to `snake_case` and categorical text fields to Title Case.
* **Missing Value & Duplicate Check:** Verified zero missing values and removed duplicate records.
* **Date Validation & Type Conversion:** Verified chronological date order (`discharge_date >= date_of_admission`) and converted dates to `datetime64[us]`.
* **Financial Filtering & Outlier Check:** Removed zero or negative billing records (`billing_amount <= 0`) and checked numerical columns for outliers.

In [4]:
import pandas as pd

df = pd.read_csv("healthcare_dataset.csv")

print("Database shape (Rows,Columns):,", df.shape)
df.head()

Database shape (Rows,Columns):, (55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df["name"] = df["name"].astype(str).str.title().str.strip()

print(df[["name"]].head(10))

                 name
0       Bobby Jackson
1        Leslie Terry
2         Danny Smith
3        Andrew Watts
4       Adrienne Bell
5       Emily Johnson
6      Edward Edwards
7  Christina Martinez
8     Jasmine Aguilar
9    Christopher Berg


In [6]:
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values)

Missing Values per Column:
name                  0
age                   0
gender                0
blood_type            0
medical_condition     0
date_of_admission     0
doctor                0
hospital              0
insurance_provider    0
billing_amount        0
room_number           0
admission_type        0
discharge_date        0
medication            0
test_results          0
dtype: int64


In [7]:
visit_subset = [
    "name",
    "age",
    "gender",
    "medical_condition",
    "date_of_admission",
    "insurance_provider",
    "discharge_date",
    "doctor",
    "hospital",
    "admission_type",
]

visit_duplicates = df[df.duplicated(subset=visit_subset, keep=False)]

print(
    f"Total duplicate visit records found across key criteria: {len(visit_duplicates)}"
)

visit_duplicates.sort_values(by=["name", "date_of_admission"]).head(10)

Total duplicate visit records found across key criteria: 1068


,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
42407,Abigail Young,41,Female,O+,Hypertension,2022-12-15,Edward Kramer,Moore-Mcdaniel,UnitedHealthcare,1983.568297,192,Elective,2023-01-13,Ibuprofen,Normal
54285,Abigail Young,41,Female,O+,Hypertension,2022-12-15,Edward Kramer,Moore-Mcdaniel,UnitedHealthcare,1983.568297,192,Elective,2023-01-13,Ibuprofen,Normal
40385,Adam Thomas,75,Male,O+,Hypertension,2022-01-02,Beverly Miller,Sanchez-Wilson,Cigna,16294.907725,180,Urgent,2022-01-30,Ibuprofen,Normal
53139,Adam Thomas,75,Male,O+,Hypertension,2022-01-02,Beverly Miller,Sanchez-Wilson,Cigna,16294.907725,180,Urgent,2022-01-30,Ibuprofen,Normal
24779,Alex Black,51,Male,O+,Diabetes,2022-03-27,Frank Williams,Short and Sons,Aetna,39525.674902,246,Emergency,2022-04-21,Aspirin,Normal
55298,Alex Black,51,Male,O+,Diabetes,2022-03-27,Frank Williams,Short and Sons,Aetna,39525.674902,246,Emergency,2022-04-21,Aspirin,Normal
31845,Alexandra Santiago,53,Male,B-,Arthritis,2024-01-17,Donald Solis,"and White, Wall Hurst",UnitedHealthcare,4177.715918,450,Urgent,2024-02-15,Penicillin,Normal
54118,Alexandra Santiago,53,Male,B-,Arthritis,2024-01-17,Donald Solis,"and White, Wall Hurst",UnitedHealthcare,4177.715918,450,Urgent,2024-02-15,Penicillin,Normal
16386,Alfred Cervantes,65,Male,AB-,Diabetes,2023-12-24,Linda Hall,Dorsey Ltd,Aetna,1903.879223,385,Elective,2024-01-04,Paracetamol,Inconclusive
54232,Alfred Cervantes,65,Male,AB-,Diabetes,2023-12-24,Linda Hall,Dorsey Ltd,Aetna,1903.879223,385,Elective,2024-01-04,Paracetamol,Inconclusive


In [8]:
df = df.drop_duplicates(subset=visit_subset, keep="first")

print(f"New dataset shape after removing duplicates: {df.shape}")

New dataset shape after removing duplicates: (54966, 15)


In [9]:
remaining_duplicates = df[df.duplicated(subset=visit_subset, keep=False)]

print(f"Total duplicate visit records remaining: {len(remaining_duplicates)}")

Total duplicate visit records remaining: 0


In [10]:
invalid_dates = df[df["discharge_date"] < df["date_of_admission"]]
print(f"Records with invalid date logic: {len(invalid_dates)}")

Records with invalid date logic: 0


In [11]:
print(df["billing_amount"].describe())

invalid_billing = df[df["billing_amount"] <= 0]
print(f"Records with zero or negative billing: {len(invalid_billing)}")

count    54966.000000
mean     25544.306284
std      14208.409711
min      -2008.492140
25%      13243.718641
50%      25542.749145
75%      37819.858159
max      52764.276736
Name: billing_amount, dtype: float64
Records with zero or negative billing: 106


In [12]:
negative_billing_records = df[df["billing_amount"] <= 0]

print(f"Total records with zero or negative billing: {len(negative_billing_records)}")

negative_billing_records.head(10)

Total records with zero or negative billing: 106


,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
132,Ashley Erickson,32,Female,AB-,Cancer,2019-11-05,Gerald Hooper,"and Johnson Moore, Branch",Aetna,-502.507813,376,Urgent,2019-11-23,Penicillin,Normal
799,Christopher Weiss,49,Female,AB-,Asthma,2023-02-16,Kelly Thompson,Hunter-Hughes,Aetna,-1018.245371,204,Elective,2023-03-09,Penicillin,Inconclusive
1018,Ashley Warner,60,Male,A+,Hypertension,2021-12-21,Andrea Bentley,"and Wagner, Lee Klein",Aetna,-306.364925,426,Elective,2022-01-11,Ibuprofen,Normal
1421,Jay Galloway,74,Female,O+,Asthma,2021-01-20,Debra Everett,Group Peters,Blue Cross,-109.097122,381,Emergency,2021-02-09,Ibuprofen,Abnormal
2103,Joshua Williamson,72,Female,B-,Diabetes,2021-03-21,Wendy Ramos,"and Huff Reeves, Dennis",Blue Cross,-576.727907,369,Urgent,2021-04-17,Aspirin,Abnormal
2696,Scott Vazquez,74,Male,B+,Diabetes,2023-04-12,Edward Yates,James Ltd,Medicare,-135.986000,445,Elective,2023-05-03,Ibuprofen,Abnormal
2855,Carol Anderson,39,Female,B-,Hypertension,2020-04-03,Dr. Patrick Hines,"Carter Carter, and Patterson",Blue Cross,-370.983674,203,Elective,2020-04-10,Ibuprofen,Abnormal
3772,Mr. Christopher Alvarado,77,Male,AB+,Obesity,2022-06-03,Mr. Dean Guzman DDS,Johnson Inc,Blue Cross,-1310.272895,257,Elective,2022-06-13,Paracetamol,Inconclusive
5445,Alexandra Khan,32,Male,AB+,Arthritis,2022-07-14,Michael Vaughn,"Bowen Lopez, and Terry",Aetna,-692.408820,372,Elective,2022-07-19,Lipitor,Abnormal
5708,Joseph Cox,23,Male,AB-,Diabetes,2019-10-13,Peter Smith,Inc Ward,Blue Cross,-353.865186,271,Elective,2019-10-25,Lipitor,Inconclusive


In [13]:
df = df[df["billing_amount"] > 0]

print(f"Final dataset shape after cleaning: {df.shape}")

Final dataset shape after cleaning: (54860, 15)


In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 54860 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                54860 non-null  str    
 1   age                 54860 non-null  int64  
 2   gender              54860 non-null  str    
 3   blood_type          54860 non-null  str    
 4   medical_condition   54860 non-null  str    
 5   date_of_admission   54860 non-null  str    
 6   doctor              54860 non-null  str    
 7   hospital            54860 non-null  str    
 8   insurance_provider  54860 non-null  str    
 9   billing_amount      54860 non-null  float64
 10  room_number         54860 non-null  int64  
 11  admission_type      54860 non-null  str    
 12  discharge_date      54860 non-null  str    
 13  medication          54860 non-null  str    
 14  test_results        54860 non-null  str    
dtypes: float64(1), int64(2), str(12)
memory usage: 12.6 MB


In [15]:
print("date_of_admission type:", df["date_of_admission"].dtype)
print("discharge_date type:", df["discharge_date"].dtype)

date_of_admission type: str
discharge_date type: str


In [16]:
df["date_of_admission"] = pd.to_datetime(df["date_of_admission"])
df["discharge_date"] = pd.to_datetime(df["discharge_date"])

In [17]:
print("date_of_admission type:", df["date_of_admission"].dtype)
print("discharge_date type:", df["discharge_date"].dtype)

date_of_admission type: datetime64[us]
discharge_date type: datetime64[us]


In [18]:
df
df.sample(10)

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
32026,Walter Jackson,54,Female,B+,Obesity,2021-05-28,Amy Sanders,Hays-Wallace,Cigna,9358.970270,293,Elective,2021-06-20,Lipitor,Abnormal
9168,Alex Williams Md,76,Male,O+,Asthma,2022-10-05,Jennifer Chapman,Phillips LLC,Blue Cross,2036.289199,353,Emergency,2022-10-20,Aspirin,Inconclusive
32427,Mary Murray,51,Male,A-,Obesity,2021-08-23,Jeffery Bailey,"Wheeler, Hernandez and Miller",UnitedHealthcare,14579.703840,298,Emergency,2021-09-11,Ibuprofen,Inconclusive
3518,Anthony Moreno,54,Male,O+,Asthma,2020-10-18,Regina Ramirez,"Jackson Contreras and Byrd,",UnitedHealthcare,16958.227412,214,Elective,2020-10-28,Penicillin,Abnormal
15633,Tina Jefferson,28,Male,O+,Asthma,2022-02-07,Daniel Bailey,Ltd Roberts,UnitedHealthcare,2535.453713,366,Elective,2022-02-23,Lipitor,Abnormal
22954,Riley Mata,48,Female,B-,Diabetes,2022-06-11,Gary Paul,Stewart Inc,Aetna,29730.587081,129,Elective,2022-06-26,Penicillin,Normal
25166,Tony Reed,52,Male,B+,Cancer,2021-07-06,Mary Yoder,Wright-Holmes,Medicare,48521.801756,115,Emergency,2021-07-23,Aspirin,Abnormal
6219,Allison Conner,43,Female,B-,Diabetes,2022-12-28,James Roman,"Murillo and Sanders, Morris",Aetna,36946.386603,328,Emergency,2023-01-14,Aspirin,Abnormal
45090,Miss Stephanie Allen Md,72,Female,O-,Arthritis,2021-07-30,John Williams,"Rogers and Taylor Garza,",UnitedHealthcare,23193.744647,311,Elective,2021-08-27,Lipitor,Inconclusive
6755,Colton Martinez,30,Female,O-,Cancer,2020-02-28,Sandra Garcia,Lewis-Smith,Blue Cross,16698.672344,433,Urgent,2020-03-23,Penicillin,Inconclusive


In [19]:
num_cols = ["age", "billing_amount", "room_number"]

print("--- Summary Statistics ---")
display(df[num_cols].describe())

print("\n--- Outlier Check (IQR Method) ---")
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers (bounds: {lower:.2f} to {upper:.2f})")

--- Summary Statistics ---


,age,billing_amount,room_number
count,54860.000000,54860.000000,54860.000000
mean,51.533850,25594.633637,301.109752
std,19.605295,14175.867041,115.217195
min,13.000000,9.238787,101.000000
25%,35.000000,13299.747940,202.000000
50%,52.000000,25593.873000,302.000000
75%,68.000000,37847.066671,400.000000
max,89.000000,52764.276736,500.000000



--- Outlier Check (IQR Method) ---
age: 0 outliers (bounds: -14.50 to 117.50)
billing_amount: 0 outliers (bounds: -23521.23 to 74668.04)
room_number: 0 outliers (bounds: -95.00 to 697.00)


### Outlier Audit 
Using the Interquartile Range (IQR) method (`Q1 - 1.5 * IQR` to `Q3 + 1.5 * IQR`) across `age`, `billing_amount`, and `room_number` over all **54,860** records:
* **0 statistical outliers** were detected in any of the three numerical columns.
* All values sit within sensible clinical, billing, and room-number ranges, so **no records were removed** at this step.

In [20]:
display(df)
print(f"Final dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

df.to_csv("cleaned_healthcare_dataset.csv", index=False)
print("Exported 'cleaned_healthcare_dataset.csv' successfully!")

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,Elizabeth Jackson,42,Female,O+,Asthma,2020-08-16,Joshua Jarvis,Jones-Thompson,Blue Cross,2650.714952,417,Elective,2020-09-15,Penicillin,Abnormal
55496,Kyle Perez,61,Female,AB-,Obesity,2020-01-23,Taylor Sullivan,Tucker-Moyer,Cigna,31457.797307,316,Elective,2020-02-01,Aspirin,Normal
55497,Heather Wang,38,Female,B+,Hypertension,2020-07-13,Joe Jacobs DVM,"and Mahoney Johnson Vasquez,",UnitedHealthcare,27620.764717,347,Urgent,2020-08-10,Ibuprofen,Abnormal
55498,Jennifer Jones,43,Male,O-,Arthritis,2019-05-25,Kimberly Curry,"Jackson Todd and Castro,",Medicare,32451.092358,321,Elective,2019-05-31,Ibuprofen,Abnormal


Final dataset shape: 54860 rows, 15 columns
Exported 'cleaned_healthcare_dataset.csv' successfully!
